# 05 FairJob Targeting

This notebook shows how the public `criteo/FairJob` dataset can be mapped into the DSP demo's retrieval-and-reranking abstraction.

FairJob is a better conceptual fit than MIND for this prototype because it already contains:
- users,
- products that behave like ads or offers,
- impression sessions with multiple shown items,
- click labels.

What it does **not** contain is an explicit ad-server targeting DSL. That layer is inferred from historical response patterns.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise RuntimeError('Could not locate repo root')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.models import Campaign, UserProfile
from data.common import read_jsonl
from data.fairjob_adapter import export_fairjob_dataset
from experiments.evaluate import evaluate_interaction_dataset

OUTPUT_DIR = REPO_ROOT / 'data' / 'generated' / 'fairjob'
export_fairjob_dataset(OUTPUT_DIR, max_impressions=12000, max_campaigns=2500, min_campaign_impressions=10)

metadata = pd.read_json(OUTPUT_DIR / 'metadata.json', typ='series')
users = [UserProfile.model_validate(row) for row in read_jsonl(OUTPUT_DIR / 'users.jsonl')]
campaigns = [Campaign.model_validate(row) for row in read_jsonl(OUTPUT_DIR / 'campaigns.jsonl')]
interactions = pd.read_parquet(OUTPUT_DIR / 'interactions.parquet')

metadata

## Mapping FairJob To The Demo Schema

The adapter uses this translation:
- `impression_id` becomes a request-scoped pseudo-user (`fj_imp_<id>`)
- `product_id` becomes `campaign_id`
- `click` remains the label
- user categorical and numeric fields become user segments
- product-level targeting criteria are inferred from training-history lift against those user segments

In [ ]:
pd.DataFrame(
    [
        {
            'FairJob field': 'impression_id',
            'Demo mapping': 'request-scoped user_id',
            'Reason': 'the label is tied to a displayed slate, not a persistent identity only',
        },
        {
            'FairJob field': 'product_id',
            'Demo mapping': 'campaign_id',
            'Reason': 'product/job offer is the item being retrieved and ranked',
        },
        {
            'FairJob field': 'cat0..cat5 and num16..num50',
            'Demo mapping': 'user segments and interests',
            'Reason': 'shared bucket vocabulary for Redis targeting',
        },
        {
            'FairJob field': 'click',
            'Demo mapping': 'relevance label',
            'Reason': 'offline ranking target inside the shown impression slate',
        },
    ]
)

## Example Exported User

The exported user profile keeps two exact-match axes (`geo` and `device`) as derived categorical partitions, then stores richer segment memberships for Redis retrieval and reranking.

In [ ]:
sample_user = users[0]
pd.json_normalize(sample_user.model_dump())

## Example Exported Campaign

Campaign targeting is inferred from historical segment lift:
- `required_segments` are the strongest always-helpful segments,
- `any_of_segments` are additional positive buckets,
- `none_of_segments` are negative buckets to be filtered in the app,
- `weights` are the reranking features learned from the same segment vocabulary.

In [ ]:
sample_campaign = next(campaign for campaign in campaigns if campaign.required_segments or campaign.any_of_segments)
pd.json_normalize(sample_campaign.model_dump())

## What The Interaction Table Represents

Every row in `interactions.parquet` is an item that was actually shown in a FairJob impression session.
That means offline quality metrics can be computed against real exposed slates instead of synthetic counterfactual labels.

In [ ]:
display(interactions.head(10))
display(
    pd.Series(
        {
            'rows': len(interactions),
            'unique_request_users': interactions['user_id'].nunique(),
            'unique_campaigns': interactions['campaign_id'].nunique(),
            'positive_rows': int(interactions['label'].sum()),
            'randomized_positive_rows': int(interactions.loc[interactions['displayrandom'] == 1, 'label'].sum()),
        }
    )
)

## Offline Evaluation

The FairJob evaluator is intentionally more conservative than the synthetic one:
- it evaluates against the observed impression slate,
- it can restrict to `displayrandom = 1` to reduce position bias,
- it reports candidate recall and displayed-slate coverage separately.

This is not the same as a full counterfactual auction replay, but it is a defensible way to compare retrieval and reranking quality on a real ad-like dataset.

In [ ]:
all_results = evaluate_interaction_dataset(OUTPUT_DIR, top_k=5, sample_users=200, randomized_only=False)
randomized_results = evaluate_interaction_dataset(OUTPUT_DIR, top_k=5, sample_users=200, randomized_only=True)

display(pd.Series(all_results, name='all_impressions'))
display(pd.Series(randomized_results, name='displayrandom_only'))